# Preprocessing

In [ ]:
import pathlib
import numpy as np
import pandas as pd
import polars as pl
from tqdm.notebook import tqdm

from navigoquest import PathDataset
from navigoquest.environments import BoundaryEnvironment, CohortEnvironment

# Import utility variables
from navigoquest.utils.config import (
    BOUNDARY_PARAMs,
    LEVELS,
    METADATA_COLS,
    METADATA_FILENAME,
    ODMAT_WINDOW_SIZE,
    ODMAT_WEIGHT_SCALE,
)

In [ ]:
# Input directory with ALL attempts (change as needed)
raw_data_dir = pathlib.Path.home() / "Documents/raw_data"

data_dir = pathlib.Path.cwd().parent / "data"

# First attempts: written by the filtering step, read by the environment steps
raw_paths_dir = data_dir / "raw_paths"

# Output directories for the environments
out_cohort_env_dir = data_dir / "env_cohort"
out_boundary_env_dir = data_dir / "env_boundary"

# We create them when they don't exist
raw_paths_dir.mkdir(parents=True, exist_ok=True)
out_cohort_env_dir.mkdir(parents=True, exist_ok=True)
out_boundary_env_dir.mkdir(parents=True, exist_ok=True)

## Filter first attempts

In [ ]:
users_df = pd.read_csv(raw_paths_dir / METADATA_FILENAME)

# NB: we remove a user that has an incorrect recording for level 2 (plus it only recorded levels 2 and 6)
user_set = set(users_df["id"]).difference({"dca0ce68-226b-4594-a341-e2f99961805d"})

N = len(user_set)
print(f"Succesfully loaded {N:,} unique users")

In [ ]:
# Column names stored in each level CSV, to pass unto polars
path_level_cols = [
    "instance_id",
    "user_id",
    "duration",
    "trajectory_data",
    "timestamp"
]

In [ ]:
for lvl in tqdm(LEVELS):
    
    input_filename = f"level_20180917_{lvl}.csv"
    output_filename = f"level{lvl:02}_gb_1998_first-attempts.csv"
    
    (
        pl.scan_csv(raw_data_dir / input_filename, has_header=False, new_columns=path_level_cols)
        .filter(pl.col("user_id").is_in(user_set))
        .sort("timestamp")  # sorting will determine what attempt is considered the "first"
        .group_by("user_id")
        .first()
        .sink_csv(raw_paths_dir / output_filename)
    )

## Compute boundary environments

In [ ]:
for lvl, params in BOUNDARY_PARAMs.items():
    print(f"\nLevel{lvl}")
    
    output_filename = f"level{lvl:02}_boundary_env.pkl"
    
    env = BoundaryEnvironment(level=lvl, scale=4.0, **params)
    env.to_pickle(out_boundary_env_dir / output_filename)

## Compute cohort environments

In [ ]:
user_metadata_path = raw_paths_dir / METADATA_FILENAME

for lvl in tqdm(LEVELS):
    print(f"\nLevel{lvl}")

    input_filename = f"level{lvl:02}_gb_1998_first-attempts.csv"
    output_filename = f"level{lvl:02}_cohort_env.pkl"

    dataset = PathDataset.from_level_csv(
        paths_filename=raw_paths_dir / input_filename,
        metadata_filename=user_metadata_path,
        metadata_keep_cols=METADATA_COLS,
    )

    env = CohortEnvironment(level=lvl)
    
    env.set_od_matrices(dataset, *METADATA_COLS)
    env.transform_od_matrices_to_windowed(half_window=ODMAT_WINDOW_SIZE, scale=ODMAT_WEIGHT_SCALE)
    env.set_mobility_fields()
    
    env.to_pickle(out_cohort_env_dir / output_filename)

## Extract map boundaries

In [ ]:
import os
# import time
# import math
import json
import numpy as np
# import pandas as pd
# import pickle
# import pyarrow.feather as feather
# from scipy.ndimage import gaussian_filter
import matplotlib.pyplot as plt
# import datetime
# from sklearn.metrics import confusion_matrix
# from sklearn.linear_model import LogisticRegression
# from sklearn.model_selection import train_test_split
# from sklearn.preprocessing import StandardScaler
# from sklearn.neural_network import MLPClassifier
# from sklearn.ensemble import RandomForestClassifier
from scipy.spatial import distance_matrix
# from sklearn.neighbors import KDTree

from pathlib import Path

In [ ]:
# Legacy code from Shqod
def _open_level_file(filename):
    # Load the data in a level json file.
    if (ext := os.path.splitext(filename)[1]) != ".json":
        raise ValueError(f"unsupported format: {ext}")

    with open(filename, "r") as f:
        data = json.loads(f.read())
        assert "fixed" in data.keys(), "error: missing key `fixed`"
        data = data["fixed"]

    return data

def read_level_grid(filename):
    # Load the boundaries and the dimension of the grid from the JSON file of a level.
    data = _open_level_file(filename)

    wd, lg = data["grid_width"], data["grid_length"]
    grid = np.array(data["grid_data"]).reshape((wd, lg), order="F")
    coords = np.vstack(grid.nonzero()).T

    return coords, wd, lg

def read_level_flags(filename):
    # Get the coordinates of the flags (checkpoints) from the JSON file of a level.
    data = _open_level_file(filename)

    return np.array([(d["x"], d["y"]) for d in data["flags"]])

In [ ]:
print("Working directory = " + os.getcwd())
maps_data_dir = Path('../../data')
internal_data_dir = Path('data_dim_red')

In [ ]:
# Read maps
lvls = [1, 2, 6, 8, 11]
maps = {}
map_dims = {}
checkpoints = {}
for lvl in lvls:
    if lvl < 10:
        lvl_str = '0'*(lvl<10) + str(lvl)
    else:
        lvl_str = str(lvl)
    filename = Path(maps_data_dir / 'maps' / f'level{lvl_str}.json')
    arr, w, l = read_level_grid(filename)
    checkpoint = read_level_flags(filename)
    maps[lvl] = np.array(arr)
    map_dims[lvl] = np.array([w, l])
    checkpoints[lvl] = np.array(checkpoint)

In [ ]:
# Display maps
for lvl in lvls:
    plt.scatter(maps[lvl][:,0], maps[lvl][:,1], c = 'gray')
    plt.scatter(checkpoints[lvl][:,0], checkpoints[lvl][:,1], c = 'red')
    plt.show()

In [ ]:
# Extract map boundary
maps_bdy = {}
for lvl in lvls:
    game_map = maps[lvl]
    N = game_map.shape[0]
    bdy_inds = []
    counter = 0
    for i in range(N):
        pt = game_map[i]
        pt_x = pt[0]
        pt_y = pt[1]
        near_pts = np.array([[pt_x+1, pt_y], [pt_x-1, pt_y], [pt_x, pt_y+1], [pt_x, pt_y-1]])
        check_val = True
        for near_pt in near_pts:
            check_arr = np.vstack([game_map[:,0] == near_pt[0], game_map[:,1] == near_pt[1]])
            check_val *= np.sum(np.prod(check_arr, axis=0))
        if not check_val:
            bdy_inds.append(i)
            counter += 1
    print(f"N = {N}, counter = {counter}")
    bdy_inds = np.array(bdy_inds)
    bdy_pts = game_map[bdy_inds]
    plt.scatter(bdy_pts[:,0], bdy_pts[:,1], s=0.5)
    plt.show()
    maps_bdy[lvl] = bdy_pts

In [ ]:
# Find the outer boundary
bad_pts_all = {}
# Include these points in, separately
except_pts = {6: [],
             8: [],
             11: [[6, 10], [29, 45], [29, 46], [29, 47], [29, 48], [29, 49], [29, 50], [29, 51], [29, 52], [29, 53],
                 [29, 54], [38, 47], [38, 48], [38, 49], [38, 50], [41, 53], [52, 35], [49, 8 ], [45, 11],
                 [26, 19], [26, 20], [26, 21], [26, 22], [26, 23], [26, 24], [26, 25],
                 [28, 17], [29, 17], [30, 17], [31, 17], [32, 17], 
                 [34, 16], [35, 16], [36, 16], [37, 16], [38, 16], 
                 [39, 15], [40, 14], [28, 27], [29, 27], [30, 27], [31, 27], [32, 27],
                 [27, 26], [45, 20], [46, 20], [47, 20],
                 [33, 26], [34, 25], [34, 24], [35, 24], [36, 24], [37, 23], [38, 22],
                 [38, 21], [39, 20], [40, 20], [41, 20], [42, 20], [43, 20], [43, 19]]}
D = 70 # Maximum size of the map
for lvl in lvls:
    bdy_pts = maps_bdy[lvl]
    xsort_inds = np.argsort(bdy_pts[:,0])
    ysort_inds = np.argsort(bdy_pts[:,1])
    xsort_pts = bdy_pts[xsort_inds]
    ysort_pts = bdy_pts[ysort_inds]
    
    bad_pts = []
    for x_val in range(D):
        inds = np.nonzero(xsort_pts[:, 1] == x_val)[0]
        if inds.size > 0:        
            min_ind = np.min(inds)
            max_ind = np.max(inds)
            bad_pts.append(xsort_pts[min_ind])
            bad_pts.append(xsort_pts[max_ind])
    
    for y_val in range(D):
        inds = np.nonzero(ysort_pts[:, 0] == y_val)[0]
        if inds.size > 0:        
            min_ind = np.min(inds)
            max_ind = np.max(inds)
            bad_pts.append(ysort_pts[min_ind])
            bad_pts.append(ysort_pts[max_ind])
            
    bad_pts = np.array(bad_pts)
    if lvl == 11:
        bad_pts = np.vstack([bad_pts, np.array(except_pts[lvl])])
    plt.scatter(bad_pts[:,0], bad_pts[:,1], s = 0.5, c = "red")
    plt.show()
    bad_pts_all[lvl] = bad_pts

In [ ]:
# Find inner boundary points
inner_bdy_pts = {}
except_inds = {1: [20,  24,  28,  29,  32,  35,  41,  42,  47,  48,  19,  23,  27,
                   87,  88,  89, 102, 107, 119],
               2: [0,   1,   2,   3,   4,   5,   6,   7,   8,   9,  10,  11,  12,
        13,  14,  15,  16,  17,  18,  19,  20,  21,  22,  23,  24,  25,
        26,  27,  28,  29,  30,  31,  32,  33,  34,  35,  36,  37,  38,
        39,  61,  62,  63,  64,  85,  86,  90,  91,  61,  62,  63,  85,
        90,  94,  98, 104, 105, 112, 113, 139, 140, 162, 163, 168, 169,
       170, 162, 168, 175, 176,  91,  95,  99, 105, 113, 116, 186, 260,
       261, 262, 263, 264, 265, 266, 267, 268, 269, 270, 271, 272, 273,
       274, 275, 276, 277, 278, 279, 280, 281, 282, 283, 284, 285, 286,
       287, 235, 236, 237, 238, 245, 246, 247, 256, 257, 258, 259],
               6: [0, 1, 2, 3, 11, 37, 38, 38, 41, 129],
              8: [26, 35, 204],
              11: []}
for lvl in lvls:
    bad_pts_now = bad_pts_all[lvl]
    bdy_pts_now = maps_bdy[lvl]
    inner_bdy_pts_now = []
    for pt in bdy_pts_now:
        check_arr = np.vstack([bad_pts_now[:,0] == pt[0], bad_pts_now[:,1] == pt[1]])
        is_inner = (np.sum(np.prod(check_arr, axis=0)) == 0)
        if is_inner:
            inner_bdy_pts_now.append(pt)
    inner_bdy_pts_now = np.array(inner_bdy_pts_now)
    
    my_inds = np.array(list(set(np.arange(0, inner_bdy_pts_now.shape[0])).difference(except_inds[lvl])))
    inner_bdy_pts_now = inner_bdy_pts_now[my_inds]
    
    inner_bdy_pts[lvl] = inner_bdy_pts_now
    
    plt.scatter(inner_bdy_pts_now[:, 0], inner_bdy_pts_now[:, 1])
    plt.show()

In [ ]:
l = 5 # how many nearest indices to compute

map_bdy_inds = {}
for lvl in lvls:
    my_pts = inner_bdy_pts[lvl]
    next_inds = []
    N = my_pts.shape[0]
    distmat = distance_matrix(my_pts, my_pts)
    distmat_ignore_diag = distmat + 1000*np.identity(N)
    nearest_many_inds = np.argsort(distmat_ignore_diag, axis=1)[:, :l] # Nearest l points indices
    
    ind_now = 0
    inds_so_far = []
    for i in range(N-1):
        if ind_now == 0:
            inds_so_far.append(ind_now)
            ind_now = nearest_many_inds[ind_now, 0]
        else:
            check_these_inds = nearest_many_inds[ind_now]
            check_counter = 0
            while True:
                candidate_ind = check_these_inds[check_counter]
                if not candidate_ind in set(inds_so_far):
                    inds_so_far.append(candidate_ind)
                    ind_now = candidate_ind
                    break
                else:
                    check_counter += 1
                if check_counter == l:
                    assert False
    map_bdy_inds[lvl] = inds_so_far

In [ ]:
# Sequentially arranged inner boundary points

inner_bdy_pts_seq = {}
for lvl in lvls:
    pts = inner_bdy_pts[lvl]
    pts_seq = pts[map_bdy_inds[lvl]]
    inner_bdy_pts_seq[lvl] = pts_seq
    if lvl == 1: # This fixes a tiny problem
        inner_bdy_pts_seq[1][1, 1] = 11
        inner_bdy_pts_seq[1][2, 1] = 12
    np.save(f"map_inner_bdy_points_sequential_level{lvl}", pts_seq)
    plt.plot(pts_seq.T[0], pts_seq.T[1])
    plt.show()

In [ ]:
# for lvl in [1, 2]:
#     ibp = inner_bdy_pts[lvl]
#     plt.scatter(ibp[:, 0], ibp[:, 1])
#     plt.show()

# premap1 = inner_bdy_pts[1]
# bad_inds1 = np.nonzero((premap1.T[0] < 17) 
#                        & (premap1.T[0] > 7) 
#                        & (premap1.T[1] < 50)
#                        & (premap1.T[1] > 30))[0]
# bad_inds2 = np.nonzero((premap1.T[0] < 15) 
#                        & (premap1.T[0] > 5) 
#                        & (premap1.T[1] < 25)
#                        & (premap1.T[1] > 20))[0]
# bad_inds3 = np.nonzero((premap1.T[0] < 24) 
#                        & (premap1.T[0] > 22) 
#                        & (premap1.T[1] < 45)
#                        & (premap1.T[1] > 37))[0]
# bad_inds4 = np.nonzero((premap1.T[0] < 25) 
#                        & (premap1.T[0] > 22) 
#                        & (premap1.T[1] < 18)
#                        & (premap1.T[1] > 14))[0]
# bad_inds5 = np.nonzero((premap1.T[0] < 30) 
#                        & (premap1.T[0] > 25) 
#                        & (premap1.T[1] < 26)
#                        & (premap1.T[1] > 18))[0]
# bad_inds6 = np.nonzero((premap1.T[0] < 37) 
#                        & (premap1.T[0] > 32) 
#                        & (premap1.T[1] < 34)
#                        & (premap1.T[1] > 30))[0]
# bad_inds_map1 = np.hstack([bad_inds1, bad_inds2, bad_inds3, bad_inds4, bad_inds5, bad_inds6])

# for lvl in [2]:
#     ibp = inner_bdy_pts[lvl]
#     plt.scatter(ibp[:, 0], ibp[:, 1])
#     plt.xticks(np.arange(2, 15)*4)
#     plt.yticks(np.arange(0, 22)*4)
#     plt.show()

# premap1 = inner_bdy_pts[2]
# bad_inds1 = np.nonzero((premap1.T[0] < 21) 
#                        & (premap1.T[0] > 0) 
#                        & (premap1.T[1] < 90)
#                        & (premap1.T[1] > 0))[0]
# bad_inds2 = np.nonzero((premap1.T[0] < 24) 
#                        & (premap1.T[0] > 20) 
#                        & (premap1.T[1] < 76)
#                        & (premap1.T[1] > 56))[0]
# bad_inds3 = np.nonzero((premap1.T[0] < 31) 
#                        & (premap1.T[0] > 20) 
#                        & (premap1.T[1] < 74)
#                        & (premap1.T[1] > 56))[0]
# bad_inds4 = np.nonzero((premap1.T[0] < 32) 
#                        & (premap1.T[0] > 28) 
#                        & (premap1.T[1] < 70)
#                        & (premap1.T[1] > 64))[0]
# bad_inds5 = np.nonzero((premap1.T[0] < 28) 
#                        & (premap1.T[0] > 20) 
#                        & (premap1.T[1] < 75)
#                        & (premap1.T[1] > 70))[0]
# bad_inds6 = np.nonzero((premap1.T[0] < 36) 
#                        & (premap1.T[0] > 24) 
#                        & (premap1.T[1] < 90)
#                        & (premap1.T[1] > 84))[0]
# bad_inds7 = np.nonzero((premap1.T[0] < 70) 
#                        & (premap1.T[0] > 44) 
#                        & (premap1.T[1] < 90)
#                        & (premap1.T[1] > 56))[0]
# bad_inds8 = np.nonzero((premap1.T[1] > 74) 
#                        & (premap1.T[0] == 42))[0]
# bad_inds9 = np.nonzero((premap1.T[1] > 72) 
#                        & (premap1.T[0] == 43))[0]
# bad_inds10 = np.nonzero((premap1.T[1] > 68) 
#                        & (premap1.T[0] == 44))[0]